# Tutorial to pull data from the [DESI Legacy survey](https://www.legacysurvey.org/) and fit it

### 1. Download the data

In [2]:
import requests
from tqdm import tqdm
import os

def get_file(pos:tuple, size:int=512, pixscale:float=0.262, band:str="i", savefolder:str="data/"):
    """
    Download images from the DESI Legacy survey to your local machine.

    pos : (ra,dec) in decimal degrees
    size : size of one side of the image you want to download, in pixels
    pixscale : the arcsecond/pixel ratio you want in the downloaded image. The real one is 0.262, so it's best not to modify it.
    band : the string representing which band you want to download (choose between g,r,i, and z) !!!Only use one at a time, each band has its own PSF!!!
    savefolder : path of the directory you want to download the data to.
    """
    ra, dec = pos
    if not os.path.exists(savefolder): # Create the savefolder if it doesn't already exist.
        os.makedirs(savefolder)
    # Get the sky image
    url = f'https://www.legacysurvey.org/viewer/fits-cutout?ra={str(ra)}&dec={str(dec)}&size={size}&layer=ls-dr10&pixscale={pixscale}&bands={band}&invvar'
    r = requests.get(url)
    open(f"{savefolder}sky_{band}.fits" , 'wb').write(r.content)
    # Get the PSF image
    url = f'https://www.legacysurvey.org/viewer/coadd-psf/?ra={str(ra)}&dec={str(dec)}&layer=ls-dr10&bands={band}'
    r = requests.get(url)
    open(f"{savefolder}PSF_{band}.fits" , 'wb').write(r.content)

In order to download all the data for a single object, loop ```get_file``` over the different bands.

In [5]:
pos = (19.7861250,-34.1916944) # Example RA and DEC of GSN 069
for band in "griz":
    get_file(pos, size=512, pixscale=0.262, band=band, savefolder="data/GSN069/")

### 2. Fit the data

In [ ]:
import numpy as np
import astropy.io.fits as pyfits
import galight.tools.astro_tools as astro_tools
from galight_modif.data_process import DataProcess
from galight_modif.fitting_specify import FittingSpecify
from galight_modif.fitting_process import FittingProcess
import copy
import lenstronomy.Util.param_util as param_util

In [ ]:
def fit_object(pos:tuple, img_path:str, psf_path:str=None, additional_components:str="AGN", pixel_scale:float=0.262, band:str="g", nsigma:int=15, radius:int=60, exp_sz_multiplier:float=1, npixels:int=5, savefolder:str="data/", threshold:float=5, fitting_level:str="deep", fixed_n_list:list=None, if_plot:bool=False):
    """
    Fit an image with a 2D Sérsic model and optional additional components.

    pos : (ra,dec) in decimal degrees
    img_path : str pointing to your sky_X.fits image, where X is the band.
    psf_path : str pointing to your PSF_X.fits image, where X is the band.
    additional_components : str representing which components you want to fit on top of the default Sérsic. Choices are: ["AGN","BULGE","BULGE_FIXED","BULGE+AGN"]
    pixel_scale : the arcsecond/pixel ratio of the downloaded image. The default one is 0.262.
    band : the string representing which band you want to fit (choose between g,r,i, and z)
    nsigma : int to choose the number of sigmas needed for a source in the sky image to be considered significant (non-background).
    radius : int representing the size (in pixels) of the cutout that will be kept of the original sky_X.fits image for the fitting.
    exp_sz_multiplier : float, optional tweak option to change the size of the detected source(s) for the modelling/masking. Leaving it at 1 usually works.
    npixels : int, minimum number of pixels that a source must span to be considered a source.
    savefolder : path of the directory you want to download the data to (plots+.pkl).
    threshold : float, determines how sensitive the program is to find PSF stars in the sky_X.fits image. Only relevant if no psf_path is provided.
    fitting_level : str, option to choose how long the PSO and the MCMC will run, usually resulting in better fits the longer it runs. Options are ["shallow","deep","mega_deep","giga_deep","paper_deep"]
    fixed_n_list : list, option to fix the Sérsic index of the various sources. (e.g., to fix the Sérsic index of the first source at 2.09: [[0,2.09]])
    if_plot : bool, flag if you want the code to plot the figures or calculate everything silently.
    """
    if additional_components is None:
        number_of_ps = 0
        bulge = False
    elif additional_components.upper() == "AGN":
        number_of_ps = 1
        bulge = False
    elif additional_components.upper() in ["BULGE", "BULGE_FIXED"]:
        number_of_ps = 0
        bulge = True
    elif additional_components.upper() in ["BULGE+AGN", "AGN+BULGE"]:
        number_of_ps = 1
        bulge = True
    else:
        raise ValueError(f"{additional_components} is not a supported fitting type")
    if band not in "griz":
        raise ValueError(f"band {band} is not a supported filter band")

    img = pyfits.open(img_path)
    fov_noise_map = None
    zp = 22.5
    fov_image = (img[0].data) #[:,:,band_index]
    header = img[0].header
    exp =  1
    exp_map = exp
    wht = img[1].data
    mean_wht = exp * (pixel_scale)**2  #The drizzle information is used to derive the mean WHT value.
    exp_map = exp * wht/mean_wht  #Derive the exposure time map for each pixel
    fov_noise_map = 1/np.sqrt(wht)


    data_process = DataProcess(fov_image = fov_image, target_pos = [pos[0], pos[1]], pos_type = 'wcs', header = header,
                            rm_bkglight = True, exptime = exp_map, if_plot=if_plot, zp = zp, fov_noise_map=fov_noise_map, mp=True)

    data_process.generate_target_materials(radius=radius, create_mask = True, nsigma=nsigma,
                                        exp_sz= exp_sz_multiplier, npixels = npixels, if_plot=if_plot, show_materials=False)

    #Create a dictionnary of all informations that could be useful for us
    coolinfos = data_process.arguments.copy()
    coolinfos["cutout_radius"] = radius
    coolinfos["pix_scale"] = pixel_scale
    print('---------------DATA PROCESS PARAMETERS-------------')
    print('target_pos:', data_process.target_pos)
    print('zero point:', data_process.zp) #zp is in the AB system and should be 22.5: https://www.legacysurvey.org/svtips/
    print('kwargs: ', data_process.arguments)
    print('---------------------------------------------------')

    if psf_path == None:
        data_process.find_PSF(radius = 30, user_option = True, threshold=threshold)  #Try this line out!
        data_process.psf_id_for_fitting = 0
    else:
        psf_img = pyfits.open(psf_path)[0].data
        fwhm = data_process.use_custom_psf(psf_img, if_plot=if_plot)
        print(f'FWHM = {np.around(fwhm*pixel_scale, decimals=3)}"')

    #Check if all the materials is given, if so to pass to the next step.
    data_process.checkout()

    if bulge:
        apertures = copy.deepcopy(data_process.apertures)
        comp_id = 0
        add_aperture0 = copy.deepcopy(apertures[comp_id])
        #This setting assigns comp0 as 'bulge' and comp1 as 'disk'
        add_aperture0.a, add_aperture0.b = add_aperture0.a/2, add_aperture0.b/2
        apertures = apertures[:comp_id] + [add_aperture0] + apertures[comp_id:]
        data_process.apertures = apertures #Pass apertures to the data_process

        #Adding a condition so that 1)the size of the bulge is within a reasonable radius range compared to the disk radius. 2) the disk component has a higher ellipticity than the bulge.
        def condition_bulgedisk(kwargs_lens, kwargs_source, kwargs_lens_light, kwargs_ps, kwargs_special, kwargs_extinction):
            logL = 0
            #note that the Comp[0] is the bulge and the Comp[1] is the disk.
            phi0, q0 = param_util.ellipticity2phi_q(kwargs_lens_light[0]['e1'], kwargs_lens_light[0]['e2'])
            phi1, q1 = param_util.ellipticity2phi_q(kwargs_lens_light[1]['e1'], kwargs_lens_light[1]['e2'])
            cond_0 = (kwargs_lens_light[0]['R_sersic'] > kwargs_lens_light[1]['R_sersic'] * 0.9)
            cond_1 = (kwargs_lens_light[0]['R_sersic'] < kwargs_lens_light[1]['R_sersic']*0.15)
            cond_2 = (q0 < q1)
            if cond_0 or cond_1 or cond_2:
                logL -= 10**15
            return logL
    else:
        condition_bulgedisk = None

    #PREPARE THE FITTING
    fit_sepc = FittingSpecify(data_process)
    
    #Prepare the fitting sequence, keywords see notes above.
    fit_sepc.prepare_fitting_seq(point_source_num = number_of_ps,
                                fix_Re_list=None,
                                fix_n_list=fixed_n_list, #To fix the Sérsic index at 2.09: [[0,2.09]]
                                fix_center = None,
                                fix_ellipticity = None,
                                manual_bounds = None, #{'lower':{'e1': -0.5, 'e2': -0.5, 'R_sersic': 0.01, 'n_sersic': 2., 'center_x': 0, 'center_y': 0},
                                                #'upper':{'e1': 0.5, 'e2': 0.5, 'R_sersic': 5, 'n_sersic': 9., 'center_x': 0, 'center_y': 0}},
                                condition=condition_bulgedisk)


    #Build up and to pass to the next step.
    fit_sepc.build_fitting_seq()
    #Pass fit_sepc to FittingProcess,
    fit_run = FittingProcess(fit_sepc, savename = savefolder, fitting_level=fitting_level) 
    #Setting the fitting approach and Run:
    fit_run.run(algorithm_list = ['PSO', 'MCMC'], setting_list = None)
    fit_run.mcmc_result_range()
    # Plot all the fitting results:
    if number_of_ps == 0:
        fit_run.plot_final_galaxy_fit(target_ID=f'{str(pos[0])+str(pos[1])}-{band}', show_plot=if_plot)
    else:
        fit_run.plot_final_qso_fit(target_ID=f'{str(pos[0])+str(pos[1])}-{band}', show_plot=if_plot)
    fit_run.coolinfos = coolinfos
    #Save the fitting class as pickle format:
    fit_run.dump_result(savefolder=savefolder)